In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix, f1_score

DATA_DIR = Path("/kaggle/input/competitions/ai-generated-text-detection-hw3-ucu-ai-2026")
BASE_DIR = Path("/kaggle/input/notebooks/oleksandrtsybrivskyi/01-eda-split-baseline")
M512_DIR = Path("/kaggle/input/notebooks/oleksandrtsybrivskyi/02-transformer-training")
M1024_DIR = Path("/kaggle/input/notebooks/oleksandrtsybrivskyi/03-error-analysis-and-ensemble")

CLASSES = ["human", "fusion", "ai"]
LABEL2ID = {c: i for i, c in enumerate(CLASSES)}

train = pd.read_csv(DATA_DIR / "train.csv")
splits = pd.read_parquet(BASE_DIR / "folds.parquet")
base_oof = pd.read_parquet(BASE_DIR / "baseline_oof_fold0.parquet")
m512_oof = pd.read_parquet(M512_DIR / "mdeberta_oof_fold0.parquet")
m1024_oof = pd.read_parquet(M1024_DIR / "mdeberta_long_oof_fold0.parquet")

assert (train["id"].to_numpy() == splits["id"].to_numpy()).all()
train["fold"] = splits["fold"].to_numpy()
train["y"] = train["label"].map(LABEL2ID).astype(np.int64)

va = train[train["fold"] == 0].reset_index(drop=True)
for name, df in [("base", base_oof), ("m512", m512_oof), ("m1024", m1024_oof)]:
    assert (va["id"].to_numpy() == df["id"].to_numpy()).all(), f"{name} OOF ID mismatch!"
    assert (va["y"].to_numpy() == df["y_true"].to_numpy()).all(), f"{name} label mismatch!"

va["pred_base"] = base_oof["y_pred"].to_numpy()
va["pred_512"] = m512_oof["y_pred"].to_numpy()
va["pred_1024"] = m1024_oof["y_pred"].to_numpy()
va["n_words"] = va["text"].str.split().str.len()

# 1. Confusion Matrix (mDeBERTa-1024)
cm = pd.DataFrame(
    confusion_matrix(va["y"], va["pred_1024"], labels=[0, 1, 2]),
    index=[f"true_{c}" for c in CLASSES],
    columns=[f"pred_{c}" for c in CLASSES],
)
print("--- Confusion Matrix (mDeBERTa-1024, Val Fold 0) ---")
print(cm)

# 2. Скільки залишкових помилок 1024 виправляють TF-IDF і m512
err = va["pred_1024"] != va["y"]
print(
    f"\nmDeBERTa-1024 errors: {err.sum()} ({err.mean()*100:.2f}%) | "
    f"TF-IDF correct: {(err & (va['pred_base'] == va['y'])).sum()} ({(err & (va['pred_base'] == va['y'])).sum()/err.sum()*100:.1f}%) | "
    f"m512 correct: {(err & (va['pred_512'] == va['y'])).sum()} ({(err & (va['pred_512'] == va['y'])).sum()/err.sum()*100:.1f}%)"
)

def compare_groups(df, col):
    rows = []
    for k, g in df.groupby(col, observed=False):
        y = g["y"].to_numpy()
        f512 = f1_score(y, g["pred_512"].to_numpy(), average="macro", zero_division=0)
        f1024 = f1_score(y, g["pred_1024"].to_numpy(), average="macro", zero_division=0)
        per = f1_score(y, g["pred_1024"].to_numpy(), average=None, labels=[0, 1, 2], zero_division=0)
        rows.append({
            col: k, "n": len(g),
            "f1_512": f512, "f1_1024": f1024, "delta": f1024 - f512,
            "H_1024": per[0], "F_1024": per[1],
        })
    return pd.DataFrame(rows).round(4)

lang_col = next(c for c in ("language", "lang") if c in va.columns)
print(f"\n--- 512 vs 1024 by {lang_col} ---")
print(compare_groups(va, lang_col).to_string(index=False))

print("\n--- 512 vs 1024 by domain ---")
print(compare_groups(va, "domain").sort_values("f1_1024").to_string(index=False))

va["len_bin"] = pd.cut(
    va["n_words"],
    bins=[0, 200, 350, 550, 900, 10**9],
    labels=["<=200w", "201-350w", "351-550w", "551-900w", ">900w"],
)
print("\n--- 512 vs 1024 by text length ---")
print(compare_groups(va, "len_bin").to_string(index=False))

--- Confusion Matrix (mDeBERTa-1024, Val Fold 0) ---
             pred_human  pred_fusion  pred_ai
true_human        12647          701       41
true_fusion         916        15993       59
true_ai              19           51    24163

mDeBERTa-1024 errors: 1787 (3.27%) | TF-IDF correct: 966 (54.1%) | m512 correct: 673 (37.7%)

--- 512 vs 1024 by lang ---
 lang     n  f1_512  f1_1024  delta  H_1024  F_1024
de-DE 10330  0.9211   0.9682 0.0470  0.9487  0.9593
en-EN 44260  0.9387   0.9591 0.0204  0.9349  0.9459

--- 512 vs 1024 by domain ---
                domain     n  f1_512  f1_1024  delta  H_1024  F_1024
          arxiv_papers  2850  0.7961   0.8284 0.0324  0.5806  0.9049
             gutenberg  1529  0.7678   0.8435 0.0758  0.7692  0.7639
blog_authorship_corpus  8241  0.8968   0.9061 0.0093  0.8696  0.8636
      euro_court_cases  5575  0.8709   0.9539 0.0830  0.9163  0.9455
             bundestag 10230  0.9214   0.9682 0.0467  0.9488  0.9594
      house_of_commons  6358  0.9649   

In [2]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

ID2LABEL = {i: c for i, c in enumerate(CLASSES)}

def text_meta_features(texts: pd.Series) -> np.ndarray:
    s = texts.fillna("")
    n_chars = s.str.len().to_numpy(dtype=np.float64)
    n_words = s.str.split().str.len().to_numpy(dtype=np.float64)
    n_lines = (s.str.count("\n") + 1).to_numpy(dtype=np.float64)
    n_upper = s.str.count(r"[A-ZÄÖÜ]").to_numpy(dtype=np.float64)
    n_punct = s.str.count(r"[.,;:!?\"'()-]").to_numpy(dtype=np.float64)
    n_digits = s.str.count(r"\d").to_numpy(dtype=np.float64)
    denom_c = np.maximum(n_chars, 1.0)
    denom_w = np.maximum(n_words, 1.0)
    return np.column_stack([
        np.log1p(n_chars),
        np.log1p(n_words),
        n_chars / denom_w,          # середня довжина слова
        np.log1p(n_lines),
        n_words / np.maximum(n_lines, 1.0),
        n_upper / denom_c,
        n_punct / denom_c,
        n_digits / denom_c,
    ])

def fast_macro_f1(y_true, y_pred):
    cm = np.bincount(y_true * 3 + y_pred, minlength=9).reshape(3, 3)
    tp = np.diag(cm)
    denom = cm.sum(axis=0) + cm.sum(axis=1)
    return float(np.mean(np.where(denom > 0, 2.0 * tp / denom, 0.0)))

def fit_multipliers(probs, y_true):
    best_f1, best_w = -1.0, (1.0, 1.0)
    for w0 in np.linspace(0.7, 1.4, 15):
        for w1 in np.linspace(0.7, 1.4, 15):
            score = fast_macro_f1(y_true, (probs * np.array([w0, w1, 1.0])).argmax(axis=1))
            if score > best_f1:
                best_f1, best_w = score, (w0, w1)
    w0_c, w1_c = best_w
    for w0 in np.linspace(max(0.4, w0_c - 0.08), w0_c + 0.08, 17):
        for w1 in np.linspace(max(0.4, w1_c - 0.08), w1_c + 0.08, 17):
            score = fast_macro_f1(y_true, (probs * np.array([w0, w1, 1.0])).argmax(axis=1))
            if score > best_f1:
                best_f1, best_w = score, (w0, w1)
    return np.array([best_w[0], best_w[1], 1.0])

test_df = pd.read_csv(DATA_DIR / "test.csv")
base_te = pd.read_parquet(BASE_DIR / "baseline_test_preds.parquet")
m512_te = pd.read_parquet(M512_DIR / "mdeberta_test_preds.parquet")
m1024_te = pd.read_parquet(M1024_DIR / "mdeberta_long_test_preds.parquet")

# Вирівнюємо test по m1024_te["id"]
te_ids = m1024_te["id"].to_numpy()
base_te = base_te.set_index("id").loc[te_ids].reset_index()
m512_te = m512_te.set_index("id").loc[te_ids].reset_index()
test_df = test_df.set_index("id").loc[te_ids].reset_index()

score_cols_b = [f"score_{c}" for c in CLASSES]
logit_cols = [f"logit_{c}" for c in CLASSES]

S_b_va, S_b_te = base_oof[score_cols_b].to_numpy(np.float64), base_te[score_cols_b].to_numpy(np.float64)
L_512_va, L_512_te = m512_oof[logit_cols].to_numpy(np.float64), m512_te[logit_cols].to_numpy(np.float64)
L_1024_va, L_1024_te = m1024_oof[logit_cols].to_numpy(np.float64), m1024_te[logit_cols].to_numpy(np.float64)

F_va = text_meta_features(va["text"])
F_te = text_meta_features(test_df["text"])
y_va = va["y"].to_numpy()

configs = {
    "2. LogReg: 1024 + TF-IDF": (
        np.hstack([L_1024_va, S_b_va]),
        np.hstack([L_1024_te, S_b_te]),
        lambda: make_pipeline(StandardScaler(), LogisticRegression(C=0.5, max_iter=1000)),
    ),
    "3. LogReg: 1024 + 512 + TF-IDF": (
        np.hstack([L_1024_va, L_512_va, S_b_va]),
        np.hstack([L_1024_te, L_512_te, S_b_te]),
        lambda: make_pipeline(StandardScaler(), LogisticRegression(C=0.5, max_iter=1000)),
    ),
    "4. HGBDT: 1024 + 512 + TF-IDF + meta": (
        np.hstack([L_1024_va, L_512_va, S_b_va, F_va]),
        np.hstack([L_1024_te, L_512_te, S_b_te, F_te]),
        lambda: HistGradientBoostingClassifier(
            max_iter=250, learning_rate=0.04, max_depth=4,
            min_samples_leaf=80, l2_regularization=2.0, random_state=42
        ),
    ),
}

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_probs, te_probs = {}, {}

for name, (X_v, X_t, make_clf) in configs.items():
    p_oof = np.zeros((len(y_va), 3), dtype=np.float64)
    p_te = np.zeros((len(te_ids), 3), dtype=np.float64)
    for tr_i, ev_i in skf.split(X_v, y_va):
        clf = make_clf()
        clf.fit(X_v[tr_i], y_va[tr_i])
        p_oof[ev_i] = clf.predict_proba(X_v[ev_i])
        p_te += clf.predict_proba(X_t) / skf.n_splits
    oof_probs[name] = p_oof
    te_probs[name] = p_te

# 5. Бленд лінійної та нелінійної мета-моделей (50/50)
p_mix_oof = 0.5 * oof_probs["3. LogReg: 1024 + 512 + TF-IDF"] + 0.5 * oof_probs["4. HGBDT: 1024 + 512 + TF-IDF + meta"]
p_mix_te = 0.5 * te_probs["3. LogReg: 1024 + 512 + TF-IDF"] + 0.5 * te_probs["4. HGBDT: 1024 + 512 + TF-IDF + meta"]
oof_probs["5. Meta-blend (LogReg + HGBDT)"] = p_mix_oof
te_probs["5. Meta-blend (LogReg + HGBDT)"] = p_mix_te

# Оцінка всіх варіантів (+ перевірка множників на найкращому)
candidates = [("1. mDeBERTa-1024 raw (ref)", va["pred_1024"].to_numpy(), m1024_te["y_pred"].to_numpy())]
for name in oof_probs:
    candidates.append((name, oof_probs[name].argmax(axis=1), te_probs[name].argmax(axis=1)))

# Додаємо тюнінг множників на найкращих імовірностях (всередині CV)
best_prob_name = max(oof_probs, key=lambda k: f1_score(y_va, oof_probs[k].argmax(axis=1), average="macro"))
oof_thr = np.zeros(len(y_va), dtype=np.int64)
mults = []
for tr_i, ev_i in skf.split(va, y_va):
    w = fit_multipliers(oof_probs[best_prob_name][tr_i], y_va[tr_i])
    mults.append(w)
    oof_thr[ev_i] = (oof_probs[best_prob_name][ev_i] * w).argmax(axis=1)
mean_w = np.mean(mults, axis=0)
candidates.append((
    f"6. Best + multipliers ({best_prob_name.split('.')[0]})",
    oof_thr,
    (te_probs[best_prob_name] * mean_w).argmax(axis=1),
))

rows = []
for name, p_va, _ in candidates:
    per = f1_score(y_va, p_va, average=None, labels=[0, 1, 2])
    rows.append({
        "model": name,
        "macro_f1": f1_score(y_va, p_va, average="macro"),
        "f1_H": per[0], "f1_F": per[1], "f1_A": per[2],
    })
res_df = pd.DataFrame(rows).round(4)
print("--- Validation Fold 0 Comparison (Nested 5-fold CV) ---")
print(res_df.to_string(index=False))

best_idx = int(res_df["macro_f1"].idxmax())
best_name, _, best_te_pred = candidates[best_idx]

sample_sub = pd.read_csv(DATA_DIR / "sample_submission.csv")
pred_map = pd.Series(best_te_pred, index=te_ids).map(ID2LABEL)
sub = pd.DataFrame({"id": sample_sub["id"], "label": sample_sub["id"].map(pred_map)})
assert len(sub) == 25_000 and (sub["id"].to_numpy() == sample_sub["id"].to_numpy()).all()
assert sub["label"].notna().all() and set(sub["label"].unique()) == set(CLASSES)

sub_path = "/kaggle/working/submission_ensemble_v2.csv"
sub.to_csv(sub_path, index=False)
print(f"\nSaved {sub_path} | Winner: {best_name} (Val Macro-F1 = {res_df.loc[best_idx, 'macro_f1']:.4f})")
print((sub["label"].value_counts(normalize=True) * 100).round(2).reindex(CLASSES))

--- Validation Fold 0 Comparison (Nested 5-fold CV) ---
                               model  macro_f1   f1_H   f1_F   f1_A
          1. mDeBERTa-1024 raw (ref)    0.9610 0.9378 0.9488 0.9965
            2. LogReg: 1024 + TF-IDF    0.9663 0.9463 0.9552 0.9972
      3. LogReg: 1024 + 512 + TF-IDF    0.9675 0.9478 0.9571 0.9976
4. HGBDT: 1024 + 512 + TF-IDF + meta    0.9698 0.9517 0.9601 0.9975
      5. Meta-blend (LogReg + HGBDT)    0.9695 0.9511 0.9598 0.9976
           6. Best + multipliers (4)    0.9702 0.9527 0.9604 0.9975

Saved /kaggle/working/submission_ensemble_v2.csv | Winner: 6. Best + multipliers (4) (Val Macro-F1 = 0.9702)
label
human     26.20
fusion    22.45
ai        51.36
Name: proportion, dtype: float64
